In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("LoadGoldToSQL") \
    .master("local[*]") \
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "com.microsoft.sqlserver:mssql-jdbc:12.6.1.jre8"
    ) \
    .config("spark.hadoop.fs.s3a.endpoint", "http://127.0.0.1:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "12345678") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

In [2]:
print(spark.sparkContext.getConf().get("spark.jars.packages"))

org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,com.microsoft.sqlserver:mssql-jdbc:12.6.1.jre8


In [4]:
gold_df = spark.read.parquet("s3a://housing-data/gold/house_price_final.parquet")

In [5]:
gold_df.write \
    .format("jdbc") \
    .option(
        "url",
        "jdbc:sqlserver://127.0.0.1:14330;databaseName=HousePriceDB;encrypt=true;trustServerCertificate=true"
    ) \
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
    .option("dbtable", "house_price") \
    .option("user", "sa") \
    .option("password", "123456") \
    .mode("overwrite") \
    .save()